# Meme Decoder Colab Pipeline

This notebook runs the MemeCap ablation pipeline on Google Colab:

- Input settings `1-5`
- Training strategies: `zero-shot`, `projector-only`, `projector-lora`
- Google Drive data/cache/output paths

Recommended first run: use the smoke-test cells with small sample counts before launching full training.

## 1. Runtime

In Colab, choose **Runtime -> Change runtime type -> GPU**. L4/A100 is best; T4 can be used with batch size 1 and gradient accumulation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Basic GPU check
!nvidia-smi

## 2. Clone Or Update The Repository

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/baoyunfan0101/meme-decoder.git"
REPO_DIR = Path("/content/meme-decoder")

if REPO_DIR.exists():
    %cd /content/meme-decoder
    !git pull
else:
    %cd /content
    !git clone $REPO_URL
    %cd /content/meme-decoder

## 3. Install Dependencies

In [ ]:
%cd /content/meme-decoder
!pip install -q -r requirements.txt

# Qwen2.5-VL support is actively updated in transformers.
!pip install -q -U transformers accelerate peft

## 4. Configure Data Paths

Expected data layout in Drive:

```text
/content/drive/MyDrive/meme-decoder-data/data/
  raw/
    memes-trainval.json
    memes-test.json
  processed/
    memes-trainval.ocr.json
    memes-test.ocr.json
```

Images are downloaded from the `url` fields and cached under `raw/`.

In [ ]:
from pathlib import Path
import os

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/meme-decoder-data")
DATA_DIR = DRIVE_PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["RAW_DIR"] = str(RAW_DIR)
os.environ["PROCESSED_DIR"] = str(PROCESSED_DIR)
os.environ["OUTPUT_DIR"] = str(OUTPUT_DIR)

print("RAW_DIR      =", RAW_DIR)
print("PROCESSED_DIR=", PROCESSED_DIR)
print("OUTPUT_DIR   =", OUTPUT_DIR)

## 5. Optional: Download The Shared Google Drive Data

Run this only if you have not already placed the data under `DRIVE_PROJECT_DIR/data`.

In [ ]:
%cd /content/meme-decoder

trainval_path = PROCESSED_DIR / "memes-trainval.ocr.json"
test_path = PROCESSED_DIR / "memes-test.ocr.json"

if trainval_path.exists() and test_path.exists():
    print("Processed data already exists; skipping folder download.")
else:
    !python -m scripts.download_data --output "$DRIVE_PROJECT_DIR"

## 6. Verify Dataset Counts

In [ ]:
import json

for path in [PROCESSED_DIR / "memes-trainval.ocr.json", PROCESSED_DIR / "memes-test.ocr.json"]:
    print(path)
    if not path.exists():
        print("  MISSING")
        continue
    data = json.loads(path.read_text(encoding="utf-8"))
    print("  count:", len(data))
    print("  keys :", sorted(data[0].keys()) if data else [])

## 7. Download/Cache Meme Images

First run the small `--limit 5` check. If it works, run the full download cell.

In [ ]:
%cd /content/meme-decoder
!python -m scripts.download_images --processed-dir "$PROCESSED_DIR" --raw-dir "$RAW_DIR" --limit 5

In [ ]:
# Full image cache. This may take a while.
%cd /content/meme-decoder
!python -m scripts.download_images --processed-dir "$PROCESSED_DIR" --raw-dir "$RAW_DIR"

## 8. Zero-Shot Smoke Test

Input setting `4` means: image + title + image captions + OCR.

In [ ]:
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode eval \
  --strategy zero-shot \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --eval-json memes-test.ocr.json \
  --eval-max-samples 20 \
  --allow-download

## 9. Projector-Only Smoke Training

This automatically creates 5 fold files from `memes-trainval.ocr.json` if they do not exist.

In [ ]:
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-only \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --epochs 1 \
  --batch-size 1 \
  --grad-accum-steps 8 \
  --eval-max-samples 20 \
  --allow-download

## 10. Projector + LoRA Smoke Training

For T4 GPUs, keep `batch-size=1`, use gradient accumulation, and start with `lora-r=8`.

In [ ]:
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-lora \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --epochs 1 \
  --batch-size 1 \
  --grad-accum-steps 8 \
  --lora-r 8 \
  --eval-max-samples 20 \
  --allow-download

## 11. Full Training Examples

Run these after the smoke tests pass.

In [ ]:
# Full projector-only training, setting 4
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-only \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --epochs 2 \
  --batch-size 1 \
  --grad-accum-steps 8 \
  --allow-download

In [ ]:
# Full projector + LoRA training, setting 4
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-lora \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --epochs 2 \
  --batch-size 1 \
  --grad-accum-steps 8 \
  --lora-r 8 \
  --allow-download

## 12. Evaluate A Trained Checkpoint

Use the checkpoint list cell to find the run name, then set `MODEL_PATH`.

In [ ]:
from pathlib import Path

ckpt_root = OUTPUT_DIR / "checkpoints"
if ckpt_root.exists():
    for path in sorted(ckpt_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)[:10]:
        print(path)
else:
    print("No checkpoints yet:", ckpt_root)

In [ ]:
# Change this to one of the paths printed above.
MODEL_PATH = str(OUTPUT_DIR / "checkpoints" / "<run_name>")
STRATEGY = "projector-lora"  # or "projector-only"
INPUT_SETTING = 4

%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode eval \
  --strategy "$STRATEGY" \
  --input-setting "$INPUT_SETTING" \
  --model-path "$MODEL_PATH" \
  --checkpoint-name best \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --eval-json memes-test.ocr.json \
  --allow-download

## 13. Optional: Zero-Shot Input Ablation 1-5

In [ ]:
%cd /content/meme-decoder
for setting in [1, 2, 3, 4, 5]:
    print("\n=== zero-shot setting", setting, "===")
    !python -m scripts.run_pipeline \
      --mode eval \
      --strategy zero-shot \
      --input-setting "$setting" \
      --processed-dir "$PROCESSED_DIR" \
      --raw-dir "$RAW_DIR" \
      --output-dir "$OUTPUT_DIR" \
      --eval-json memes-test.ocr.json \
      --eval-max-samples 50 \
      --allow-download